In [22]:
from MDMC.MD import *
# Define the unique atoms using the ForceField atom_type
# These can be seen in the oplsaa.dat file (MDMC/MD/force_fields/data/oplsaa.dat)
# The H1 atom will be copied after the bond and bond angles have been defined
HC1 = Atom('H', position=[-0.7006,  0.3636,  0.8900], name='98', charge=0., atom_type=1)
C = Atom('C', position=[-0.3366, -0.1504,  0.0000], name='99', charge=0., atom_type=2)
O = Atom('O', position=[ 1.0849, -0.1713,  0.0000], name='96', charge=0., atom_type=3)
HO = Atom('H', position=[ 1.3606,  0.7699,  0.0000], name='97', charge=0., atom_type=4)

In [23]:
# Create the bonds with harmonic potentials
# constrain the C-H bonds as it should not change too much
CH_bond = Bond(C, HC1, constrained=True)
CO_bond = Bond(C, O)
OH_bond = Bond(O, HO)

# Create the H-C-O and H-O-C bond angles
HCO_angle = BondAngle((HC1, C, O))
HOC_angle = BondAngle((HO, O, C))

# Create the H-C-O-H dihedral
HCOH_dihedral = DihedralAngle((HC1, C, O, HO))

# Duplicate the HC1 atom (will duplicate all its bonds, bondangles and dihedrals)
HC2 = HC1.copy(position=[-0.7006,  0.3636, -0.8900])

# Create an HCH bond angle
HCH_angle = BondAngle((HC1, C, HC2))

# Duplicate the HC1 atom again
# This atom will have all bond (CH_bond) and bond angles (HCO_angle and HCH_angle) defined
HC3 = HC1.copy(position=[-0.7076, -1.1754,  0.0000])

# Create the methanol Molecule
methanol = Molecule(atoms=[HC1, HC2, HC3, C, O, HO])

# Create a universe and add the methanol
universe = Universe(dimensions=15.0, constraint_algorithm=Shake(1e-5, 100), electrostatic_solver=PPPM(accuracy=1e-4))
universe.add_structure(methanol)

# Add dispersion interactions (no dispersion for HO, so the parameter values should be 0 once the force field is applied)
HC_disp = Dispersion(universe, (1, 1), cutoff = 6.0, vdw_tail_correction=True)
C_disp = Dispersion(universe, (2, 2), cutoff = 6.0, vdw_tail_correction=True)
O_disp = Dispersion(universe, (3, 3), cutoff = 6.0, vdw_tail_correction=True)
HO_disp = Dispersion(universe, (4, 4), cutoff = 6.0, vdw_tail_correction=True)

Universe created with:
  Dimensions       [15.0, 15.0, 15.0]
  Force field                    None
  Number of atoms                   0



In [24]:
universe.add_force_field('OPLSAA')


In [25]:
for interaction in universe.nonbonded_interactions:
    interaction.cutoff=6.0

In [26]:
universe.fill(methanol, num_density=0.002)

In [27]:
universe.n_atoms

12

In [16]:
from MDMC.gui import view
view(universe)

KeyError: 5

In [5]:
simulation = Simulation(universe, engine='lammps', time_step=1., temperature=300.,
                        pressure=101325., traj_step=10, thermostat='nose',
                        barostat='nose', t_damp=100, p_damp=1000)


LAMMPS (29 Sep 2021 - Update 3)
LAMMPS output is captured by PyLammps wrapper
OMP_NUM_THREADS environment is not set. Defaulting to 1 thread. (src/comm.cpp:98)
  using 1 OpenMP thread(s) per MPI task
LAMMPS (29 Sep 2021 - Update 3)
OMP_NUM_THREADS environment is not set. Defaulting to 1 thread. (src/comm.cpp:98)
  using 1 OpenMP thread(s) per MPI task
LAMMPS output is captured by PyLammps wrapper
Total wall time: 0:00:00
Simulation created with lammps engine and settings:
  temperature     300.0
  pressure     101325.0
  thermostat       nose
  barostat         nose
  t_damp            100
  p_damp           1000



In [6]:
simulation.minimize(n_steps=100)

Exception: ERROR: KSpace style is incompatible with Pair style (src/kspace.cpp:212)